## **Income Prediction Using XGBoost Classifier with Hyperparameter Tuning**

### **Problem Statement**
The income classification problem involves predicting whether an individual
earns more or less than $50K annually based on demographic and employment
attributes such as age, education, occupation, marital status and work hours.

XGBoost (Extreme Gradient Boosting) is one of the most powerful and
widely-used ML algorithms in industry and data science competitions.
It builds trees sequentially with gradient boosting optimization —
delivering superior accuracy, speed and regularization compared to
traditional boosting methods. This project applies XGBoost on the
Adult Census Dataset (32,561 records) to build a highly accurate
income prediction model.


### **Objective**
- Predict income category (<=50K or >50K) using XGBoost Classifier

- Handle missing values encoded as ' ?' in the Adult dataset
- Apply One-Hot Encoding with drop_first=True to avoid multicollinearity
- Tune n_estimators, max_depth, learning_rate, subsample and
  colsample_bytree using GridSearchCV
- Evaluate using Accuracy, Precision, Recall, F1 and Classification Report
- Compare Before and After tuning performance

### 1. Import Libraries

In [39]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

### 2. Load Dataset

In [40]:

df = pd.read_csv("Adult_Dataset.csv")

### 3. Data Understanding

In [41]:
df.head()

,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
0,90,?,77053,HS-grad,9,Widowed,?,Not-in-family,White,Female,0,4356,40,United-States,<=50K
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K
2,66,?,186061,Some-college,10,Widowed,?,Unmarried,Black,Female,0,4356,40,United-States,<=50K
3,54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States,<=50K
4,41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States,<=50K


In [42]:
df.shape

(32561, 15)

In [43]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             32561 non-null  int64 
 1   workclass       32561 non-null  object
 2   fnlwgt          32561 non-null  int64 
 3   education       32561 non-null  object
 4   education.num   32561 non-null  int64 
 5   marital.status  32561 non-null  object
 6   occupation      32561 non-null  object
 7   relationship    32561 non-null  object
 8   race            32561 non-null  object
 9   sex             32561 non-null  object
 10  capital.gain    32561 non-null  int64 
 11  capital.loss    32561 non-null  int64 
 12  hours.per.week  32561 non-null  int64 
 13  native.country  32561 non-null  object
 14  income          32561 non-null  object
dtypes: int64(6), object(9)
memory usage: 3.7+ MB


In [44]:
df.describe()

,age,fnlwgt,education.num,capital.gain,capital.loss,hours.per.week
count,32561.000000,3.256100e+04,32561.000000,32561.000000,32561.000000,32561.000000
mean,38.581647,1.897784e+05,10.080679,1077.648844,87.303830,40.437456
std,13.640433,1.055500e+05,2.572720,7385.292085,402.960219,12.347429
min,17.000000,1.228500e+04,1.000000,0.000000,0.000000,1.000000
25%,28.000000,1.178270e+05,9.000000,0.000000,0.000000,40.000000
50%,37.000000,1.783560e+05,10.000000,0.000000,0.000000,40.000000
75%,48.000000,2.370510e+05,12.000000,0.000000,0.000000,45.000000
max,90.000000,1.484705e+06,16.000000,99999.000000,4356.000000,99.000000


In [45]:
print(df.isnull().sum())

age               0
workclass         0
fnlwgt            0
education         0
education.num     0
marital.status    0
occupation        0
relationship      0
race              0
sex               0
capital.gain      0
capital.loss      0
hours.per.week    0
native.country    0
income            0
dtype: int64


### 4. Data Preprocessing
#### 4.1 Handle Missing Values

In [46]:
df.replace(' ?', np.nan, inplace=True)
df.dropna(inplace=True)



#### 4.2 Encoding

In [47]:
df = pd.get_dummies(df, drop_first=True, dtype=int)

### 5. Model Building
#### 5.1 Split Features & Target

In [48]:
X = df.drop('income_>50K', axis=1)
y = df['income_>50K']

#### 5.2 Train-Test Split

In [49]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

#### 5.3 Model Before Tuning
- #### Model Selection

In [50]:
model_before = XGBClassifier(random_state=42, verbosity=0)

- #### Train Model

In [51]:
model_before.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, ...)

- #### Make Predictions

In [52]:
y_pred_before = model_before.predict(X_test)

- ####  Model Evaluation

In [53]:
print("BEFORE TUNING")
print("Accuracy  :", accuracy_score(y_test, y_pred_before))
print("Precision :", precision_score(y_test, y_pred_before))
print("Recall    :", recall_score(y_test, y_pred_before))
print("F1 Score  :", f1_score(y_test, y_pred_before))

BEFORE TUNING
Accuracy  : 0.8725625671733456
Precision : 0.7772549019607843
Recall    : 0.6447625243981783
F1 Score  : 0.7048364153627311


#### 5.4 Hyperparameter Tuning
- #### Define Hyperparameters

In [54]:
params = {
    'n_estimators' : [50, 100, 200],
    'max_depth'    : [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample'    : [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

- #### Apply GridSearchCV

In [55]:
grid = GridSearchCV(
    XGBClassifier(random_state=42, verbosity=0),
    param_grid=params,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

#### 5.5 Model After Tuning
- #### Train with GridSearchCV

In [56]:
grid.fit(X_train, y_train)

GridSearchCV(cv=5,
             estimator=XGBClassifier(base_score=None, booster=None,
                                     callbacks=None, colsample_bylevel=None,
                                     colsample_bynode=None,
                                     colsample_bytree=None, device=None,
                                     early_stopping_rounds=None,
                                     enable_categorical=False, eval_metric=None,
                                     feature_types=None, feature_weights=None,
                                     gamma=None, grow_policy=None,
                                     importance_type=None,
                                     interaction_constraints=Non...
                                     max_delta_step=None, max_depth=None,
                                     max_leaves=None, min_child_weight=None,
                                     missing=nan, monotone_constraints=None,
                                     multi_strategy=None, n_estimators=None,
                                     n_jobs=None, num_parallel_tree=None, ...),
             n_jobs=-1,
             param_grid={'colsample_bytree': [0.8, 1.0],
                         'learning_rate': [0.01, 0.1, 0.2],
                         'max_depth': [3, 5, 7], 'n_estimators': [50, 100, 200],
                         'subsample': [0.8, 1.0]},
             scoring='accuracy')

- #### Build Best Model

In [57]:
best_model = grid.best_estimator_

- #### Make Predictions

In [58]:
y_pred_after = best_model.predict(X_test)

- #### Model Evaluation

In [59]:
print("AFTER TUNING")
print("Best Parameters :", grid.best_params_)
print("Accuracy  :", accuracy_score(y_test, y_pred_after))
print("Precision :", precision_score(y_test, y_pred_after))
print("Recall    :", recall_score(y_test, y_pred_after))
print("F1 Score  :", f1_score(y_test, y_pred_after))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_after))

AFTER TUNING
Best Parameters : {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 200, 'subsample': 1.0}
Accuracy  : 0.8767081222171043
Precision : 0.7959677419354839
Recall    : 0.6421600520494469
F1 Score  : 0.7108390349297803

Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.95      0.92      4976
           1       0.80      0.64      0.71      1537

    accuracy                           0.88      6513
   macro avg       0.85      0.80      0.82      6513
weighted avg       0.87      0.88      0.87      6513



### **Key Insights**
- Accuracy improved from 87.25% → 87.67% after hyperparameter tuning 

- Precision improved from 0.777 → 0.796 
- F1 Score improved from 0.705 → 0.711
- Best Parameters: colsample_bytree=0.8, learning_rate=0.1,
  max_depth=5, n_estimators=200, subsample=1.0
- XGBoost outperformed all previous models:
  Decision Tree (~85%) < Random Forest (~86%) < AdaBoost (~86.4%) < XGBoost (87.67%) 
- colsample_bytree=0.8 → uses 80% of features per tree (reduces overfitting)
- subsample=1.0 → uses all training samples for best accuracy
- learning_rate=0.1 with n_estimators=200 → optimal balance of speed & accuracy
- Class imbalance present: 4976 (<=50K) vs 1537 (>50K)
- Weighted avg F1 Score = 0.87 → strongest model performance 
- XGBoost is the best performing model in this income prediction portfolio
- GridSearchCV with 5-fold CV used to find optimal parameters

### **Conclusion**
The XGBoost Classifier successfully predicted individual income categories
with the highest accuracy of 87.67% after hyperparameter tuning —
an improvement from the 87.25% baseline and the best result across
all 4 ensemble models tested.

XGBoost's gradient boosting optimization, built-in regularization and
efficient tree building gave it an edge over Decision Tree (~85%),
Random Forest (~86%) and AdaBoost (~86.4%). GridSearchCV found the
optimal combination of n_estimators=200, learning_rate=0.1, max_depth=5,
subsample=1.0 and colsample_bytree=0.8.

The Classification Report confirmed strong performance with weighted
average F1 Score of 0.87 — the highest in this portfolio. This project
demonstrates why XGBoost is the go-to algorithm for tabular data
classification in real-world ML applications.